# Hybrid Architecture Decision Report

A hybrid approach combines retrieved evidence with learned or prompted behavior. Here the model receives both policy context and style examples, then returns a structured comparison that supports an architecture recommendation.


## What A Strategy Decision Must Contain

A useful architecture decision is not just a winner label. It should say what evidence was tested, which risks remain, and what trigger would justify revisiting the decision. For RAG and fine-tuning choices, the most important risks are freshness, citation auditability, behavior consistency, cost, latency, and maintenance effort.

The hybrid strategy is often attractive, but it should not be the default just because it sounds complete. Start with the simplest approach that satisfies the evaluation criteria. Add style conditioning or fine-tuning only when the measured failures point to behavior adaptation rather than missing evidence.


## Decision Criteria Definitions

**Freshness** means the system can reflect updated policy without retraining. RAG usually wins here because documents can be updated and re-indexed. **Auditability** means a reviewer can trace an answer back to evidence. Citations and source IDs support this. **Behavior consistency** means the system responds with the same tone, format, and refusal style across similar requests. Fine-tuning or strong examples can help here.

**Latency** is the time needed to answer. Retrieval and larger prompts can add overhead. **Cost** includes provider calls, embeddings, vector storage, training, and maintenance. **Maintenance effort** includes keeping documents current, reviewing examples, monitoring failures, and deciding when a strategy should be revisited.


### Exercise

Which risk does citation auditability address: freshness, grounding, or tone?


### Solution

Citation auditability mainly addresses grounding because reviewers can trace an answer back to retrieved evidence.


## Visual Map: Hybrid Strategy Combines Evidence And Behavior

The decision report compares retrieval quality, behavior consistency, operational cost, and the trigger for a future fine-tuning experiment.

**Embedding space visualization.** Source: [Hugging Face cookbook images](https://huggingface.co/datasets/huggingface/cookbook-images/blob/main/PaCMAP_embeddings.png). This visual helps explain why retrieval evaluation is not only about the final answer: the geometry of examples and passages influences what evidence is available to the generator.

<img src="assets/hf-cookbook-pacmap-embeddings.png" alt="Embedding visualization" style="max-width: 760px; width: 100%; height: auto;">


### Exercise

What two signals does the hybrid strategy combine?


### Solution

It combines retrieved evidence for grounding with style examples for response behavior.


In [ ]:
import json
from pprint import pprint
from typing import Any

from pydantic import BaseModel, Field, ValidationError

from strategy_helpers import (
    CONFIG,
    DOCUMENTS,
    REQUESTS,
    STYLE_EXAMPLES,
    call_model,
    compact_style_examples,
    format_context,
    retrieve,
    style_messages,
)

## Load Shared Data And Runtime Helpers

All notebooks use the same fixtures and provider gateway so the comparisons stay controlled. The repeated setup lives in [`strategy_helpers.py`](strategy_helpers.py); lesson cells can now focus on the architecture decision being tested. Dry mode remains deterministic when no provider key is available.


In [ ]:
print(
    f"Loaded {len(DOCUMENTS)} documents, {len(REQUESTS)} requests, "
    f"and {len(STYLE_EXAMPLES)} style examples."
)
print(
    f"Provider: {CONFIG.provider} | model: {CONFIG.model} | dry run: {CONFIG.dry_run}"
)

## Reuse The Evaluated Retrieval Baseline

The hybrid comparison reuses the same ranked-search helper as the RAG-only notebook. Keeping retrieval fixed means later differences come from behavior conditioning rather than a different evidence path.


In [ ]:
retrieval_preview = retrieve(REQUESTS[0]["question"], top_k=3)
[(row["doc_id"], round(row["score"], 3)) for row in retrieval_preview]

## Reuse Approved Style Examples

The shared helper converts approved examples into alternating user and assistant messages. This keeps the behavior signal identical to the fine-tuning-style baseline.


In [ ]:
few_shot_preview = style_messages(STYLE_EXAMPLES[:2])
[(message["role"], message["content"][:60]) for message in few_shot_preview]

## Build The Hybrid Prompt

The hybrid prompt includes the top retrieved policy records plus examples of the desired response behavior. This is the pattern to compare against RAG-only and style-only prompts.


The hybrid result schema stores separate grounding and style scores so the decision report does not collapse everything into one vague quality label.


In [ ]:
class StrategyResult(BaseModel):
    """Structured comparison record for one strategy output."""

    strategy: str
    answer: str
    source_ids: list[str]
    grounding_score: float = Field(ge=0, le=1)
    style_score: float = Field(ge=0, le=1)
    recommendation_note: str

Build the hybrid prompt with both retrieved evidence and style guidance. The evidence stays closest to the question.


In [ ]:
def hybrid_messages(
    request: dict[str, Any], evidence: list[dict[str, Any]]
) -> list[dict[str, str]]:
    """Create a prompt that combines style examples with citeable context."""
    context = format_context(evidence)
    return [
        {
            "role": "system",
            "content": "Use the style examples and answer only from context. Return JSON matching: strategy, answer, source_ids, grounding_score, style_score, recommendation_note.",
        },
        {
            "role": "user",
            "content": f"Style examples:\n{compact_style_examples(STYLE_EXAMPLES)}",
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {request['question']}",
        },
    ]

The dry response mirrors the fields expected from the provider, including a grounding score based on whether retrieval found the expected document.


In [ ]:
def dry_hybrid_response(request: dict[str, Any], evidence: list[dict[str, Any]]) -> str:
    """Create deterministic structured output for credential-free execution."""
    retrieved_ids = [item["doc_id"] for item in evidence]
    return json.dumps(
        {
            "strategy": "hybrid",
            "answer": f"{evidence[0]['text']} [{evidence[0]['doc_id']}]",
            "source_ids": [evidence[0]["doc_id"]],
            "grounding_score": 1.0 if request["expected_doc"] in retrieved_ids else 0.5,
            "style_score": 0.8,
            "recommendation_note": "Hybrid is useful when citations and repeatable support style are both required.",
        }
    )

The final helper calls the provider and validates structured output, with an explicit fallback for parse failures.


In [ ]:
def hybrid_answer(request: dict[str, Any]) -> StrategyResult:
    """Call the provider with both retrieved evidence and style examples."""
    evidence = retrieve(request["question"], top_k=3)
    # The live call sees both style guidance and retrieved policy context.
    raw = call_model(
        hybrid_messages(request, evidence),
        dry_response=dry_hybrid_response(request, evidence),
        response_format={"type": "json_object"},
    )

    try:
        return StrategyResult.model_validate_json(raw)
    except ValidationError:
        return StrategyResult(
            strategy="hybrid",
            answer=raw,
            source_ids=[],
            grounding_score=0.0,
            style_score=0.0,
            recommendation_note="Could not parse structured provider output.",
        )

Run one hybrid call before building the strategy table so the output shape is visible.


In [ ]:
result = hybrid_answer(REQUESTS[2])
pprint(result.model_dump())

In [ ]:
# Exercise: inspect which source ID the hybrid answer used.
# hybrid_source_ids = ...


In [ ]:
# Solution
hybrid_source_ids = result.source_ids
print(hybrid_source_ids)

## Compare Strategies With Real Calls Available

The scoring table separates quality signals from operational assumptions. In dry mode, quality values are deterministic; with provider keys, the hybrid answer comes from the model.


## Interpreting The Comparison Table

The comparison table is intentionally small, but it models a real architecture review habit: keep different quality signals separate until the decision step. Retrieval grounding answers “did we bring the right evidence?” Style score answers “did the response match the desired behavior?” Operational notes answer “is this strategy maintainable enough to start with?”

If the table shows strong grounding and weak style, improve prompts or examples before scheduling a fine-tuning project. If the table shows weak grounding, fix retrieval before changing model behavior. Fine-tuning should be a response to measured behavior failures, not a shortcut around missing evidence.


## Example Strategy Recommendations

If retrieval grounding is high and style is acceptable, recommend RAG-only because it is simpler and easier to maintain. If retrieval grounding is high but tone is inconsistent, recommend a hybrid prompt with better examples before scheduling fine-tuning. If retrieval grounding is low, recommend improving chunking, query rewriting, or the corpus before changing model behavior.

Fine-tuning becomes a stronger recommendation only after repeated behavior failures are measured on held-out examples. Even then, it should be compared against a stronger prompt baseline. The decision report should explain what was tried, why the chosen path is sufficient for now, and what signal would trigger the next experiment.


In [ ]:
def retrieval_grounding_score(request: dict[str, Any], top_k: int = 3) -> float:
    """Score whether retrieval includes the expected source."""
    retrieved_ids = [
        item["doc_id"] for item in retrieve(request["question"], top_k=top_k)
    ]
    return 1.0 if request["expected_doc"] in retrieved_ids else 0.0


# Compare strategy signals per held-out request rather than relying on one example.
strategy_rows = []
for request in REQUESTS:
    hybrid = hybrid_answer(request)
    strategy_rows.append(
        {
            "request_id": request["request_id"],
            "rag_grounding": retrieval_grounding_score(request),
            "style_only_grounding": 0.0,
            "hybrid_grounding": hybrid.grounding_score,
            "hybrid_style": hybrid.style_score,
        }
    )
pprint(strategy_rows)

In [ ]:
# Exercise: count how many held-out requests were compared.
# compared_request_count = ...


In [ ]:
# Solution
compared_request_count = len(strategy_rows)
print(compared_request_count)

## Produce The Decision Report

The decision should state a starting architecture, the evidence for that choice, and what would justify moving to actual fine-tuning later.


Build decision-review messages separately from the dry response so the report structure is visible.


In [ ]:
decision_messages = [
    {
        "role": "system",
        "content": "You are an LLM architecture reviewer. Return concise JSON with recommended_strategy, rationale, and revisit_trigger.",
    },
    {
        "role": "user",
        "content": f"Use these strategy rows to decide whether to start with RAG-only, fine-tuning-only, or hybrid:\n{json.dumps(strategy_rows, indent=2)}",
    },
]

The dry decision summarizes the intended architectural conclusion.


In [ ]:
dry_decision = json.dumps(
    {
        "recommended_strategy": "RAG-first, then hybrid if style evaluation fails",
        "rationale": "Fresh policy evidence and citations are mandatory. Style adaptation is useful, but should be added only if RAG prompts do not meet tone and refusal expectations.",
        "revisit_trigger": "Collect enough labeled style failures to justify a real LoRA or hosted fine-tuning experiment.",
    },
    indent=2,
)

Call the provider or dry-mode fallback for the final decision report.


In [ ]:
raw_decision = call_model(
    decision_messages,
    dry_response=dry_decision,
    response_format={"type": "json_object"},
)
print(raw_decision)

### Exercise

What evidence would justify moving from prompt improvements to real fine-tuning?


### Solution

Repeated style or formatting failures on held-out evaluation examples would justify a real fine-tuning experiment.


In [ ]:
next_experiment = {
    "keep": "RAG evidence path",
    "try_before_fine_tuning": [
        "stronger system prompt",
        "more few-shot style examples",
        "structured response schema",
    ],
    "fine_tune_when": "style failures persist on held-out evaluation despite prompt improvements",
}
pprint(next_experiment)

## Further Reading

- [Google Cloud RAG infrastructure guidance](https://cloud.google.com/architecture/rag-capable-gen-ai-applications) for production architecture considerations.
- [NIST AI Risk Management Framework](https://www.nist.gov/itl/ai-risk-management-framework) for documenting operational risks and review triggers.
